# HAMBEE · EVA4 de Machine Learning

Reproducción del servicio y de las pruebas de inferencia en CPU. No requiere GPU ni repetir el entrenamiento.

**Antes de ejecutar:** conserva el ZIP que acompaña a este notebook. La primera celda lo solicitará si todavía no está en la sesión.

In [ ]:
from pathlib import Path
import subprocess
import sys
import zipfile

archive = Path("EVA4_RHAGUER_YMALDONADO.zip")
if not archive.is_file():
    from google.colab import files
    files.upload()
if not archive.is_file():
    raise FileNotFoundError(f"Carga el archivo {archive.name}.")

work = Path("hambee-eva4-colab").resolve()
work.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as package:
    for member in package.infolist():
        target = (work / member.filename).resolve()
        if not target.is_relative_to(work):
            raise ValueError(f"Ruta inválida en el ZIP: {member.filename}")
        package.extract(member, work)
project = work / "EVA4"
sample = sorted((project / "validacion/test/P").glob("*.jpg"))[0]
runtime = work / ".venv"
python = runtime / ("Scripts/python.exe" if sys.platform == "win32" else "bin/python")
if not python.exists():
    subprocess.run([sys.executable, "-m", "venv", str(runtime)], check=True)
subprocess.run([str(python), "-m", "pip", "install", "-q", "-r",
                str(project / "requirements-test.txt")], check=True)

def run(script, *args):
    result = subprocess.run([str(python), script, *map(str,args)], cwd=project,
                            check=True, capture_output=True, text=True, encoding="utf-8")
    return result.stdout

print("Entorno listo. Muestra:", sample.name)

## 1. Contrato de FastAPI

La prueba ejecuta solicitudes dentro del proceso con el cliente de pruebas de FastAPI. Comprueba predicción, formato inválido, archivo demasiado grande y campo obligatorio ausente. La verificación Docker se ejecuta en GitHub Actions y su evidencia se conserva en el proyecto.

In [ ]:
import json
from IPython.display import display
from PIL import Image

contract = json.loads(run("smoke.py", sample))
display(Image.open(sample).copy())
print(json.dumps(contract, indent=2, ensure_ascii=False))

## 2. Validación y comparación de latencia

El benchmark usa los 71 originales de prueba, verifica sus huellas y mide 355 ejecuciones por configuración. Los tiempos corresponden al runtime de esta sesión. La velocidad puede variar por la asignación de CPU de Colab.

In [ ]:
benchmark_path = work / "benchmark-colab.json"
run("benchmark.py", project / "validacion", "--output", benchmark_path)
results = json.loads(benchmark_path.read_text(encoding="utf-8"))
assert results["identical_predictions"]
assert results["optimizado"]["macro_f1"] >= .94
print(f"Exactitud: {results['optimizado']['accuracy']:.2%}")
print(f"F1 macro: {results['optimizado']['macro_f1']:.4f}")
print(f"p95 optimizado: {results['optimizado']['p95_ms']:.2f} ms")
print("El límite de rendimiento de CI de 25 ms requiere un ejecutor comparable.")

In [ ]:
import matplotlib.pyplot as plt

ref, opt = results["sin_optimizacion"], results["optimizado"]
fig, (latency, matrix) = plt.subplots(1, 2, figsize=(10, 4), layout="constrained")
for metric, offset in [("p50_ms", 0), ("p95_ms", 1)]:
    bars = latency.bar([offset - .18, offset + .18], [ref[metric], opt[metric]], .32,
                       color=["#70818e", "#b72438"])
    latency.bar_label(bars, fmt="%.2f", padding=3)
latency.set_title("Inferencia en CPU")
latency.set_ylabel("Milisegundos")
latency.set_xticks([0,1],["p50","p95"])
latency.set_ylim(0, max(ref["p95_ms"],opt["p95_ms"])*1.24)
from matplotlib.patches import Patch
latency.legend(handles=[Patch(color="#70818e",label="Sin optimización"),
                        Patch(color="#b72438",label="Optimizado")], frameon=False)
confusion = opt["confusion_np_p"]
matrix.imshow(confusion, cmap="Reds", vmin=0, vmax=40)
matrix.set_title("Matriz de confusión")
matrix.set_xticks([0,1],["NP","P"])
matrix.set_yticks([0,1],["NP","P"])
matrix.set_xlabel("Clase predicha")
matrix.set_ylabel("Clase real")
for i,row in enumerate(confusion):
    for j,count in enumerate(row):
        matrix.text(j,i,str(count),ha="center",va="center",
                    color="white" if count>20 else "black")
plt.show()

## 3. Tamaño de lote y capacidad

El experimento agrupa tensores precargados e informa el p95 del lote completo y las imágenes por segundo. La latencia medida excluye la espera de cola, la decodificación y HTTP.

In [ ]:
batch_path = work / "lotes-colab.json"
run("benchmark_lotes.py", project / "validacion", "--output", batch_path)
batches = json.loads(batch_path.read_text(encoding="utf-8"))
assert batches["identical_predictions"]
fig,ax = plt.subplots(figsize=(7,3.4),layout="constrained")
ax.plot([item["batch_size"] for item in batches["batch_metrics"]],
        [item["images_per_second"] for item in batches["batch_metrics"]],
        marker="o", color="#b72438")
ax.set(xlabel="Imágenes por lote", ylabel="Imágenes por segundo", title="Capacidad en CPU")
ax.set_xticks([1,2,4,8,16])
ax.grid(alpha=.2)
plt.show()

## 4. Probar otra imagen (opcional)

Escribe el nombre de un JPG o PNG disponible en la sesión. Si dejas la variable en blanco, se vuelve a usar la muestra integrada.

In [ ]:
image_path = ""  # Ejemplo: "/content/mi_abeja.jpg"
selected = Path(image_path).resolve() if image_path else sample
if not selected.is_file():
    raise FileNotFoundError(selected)
prediction = json.loads(run("smoke.py", selected))["prediction"]
display(Image.open(selected).copy())
print(json.dumps(prediction, indent=2, ensure_ascii=False))